# Laboratorio — Robot de entregas en un almacen

A partir de la **imagen**, construye el MDP y resuelvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterias / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinamica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estanteria, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas lineas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstaculos;
- tipos de piso;
- función de transición.


In [ ]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # Estado inicial (fila, columna), segun la imagen: robot en (0,0)
        self.start = (0, 0)

        # Estanterias / paredes (celdas grises con cajas): el robot no puede
        # entrar en ellas; si lo intenta, se queda en el mismo estado.
        self.walls = {
            (0, 3),
            (1, 1),
            (2, 4),
            (4, 2),
        }

        # Celdas de piso resbaloso (patrón amarillo tipo panal en la imagen)
        self.slippery_states = {
            (1, 2),
            (2, 1),
            (3, 3),
        }

        # Estados terminales: (fila, columna) -> recompensa
        self.terminal_states = {
            (0, 5): +10.0,   # zona de entrega ("ENTREGA +10")
            (2, 2): +2.0,    # estación de carga ("CARGA +2 · TERMINAL")
            (3, 5): -10.0,   # peligro mortal
        }

        # Peligros no terminales (triangulos rojos "-3" en la imagen)
        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        row, col = state
        if not (0 <= row < self.height and 0 <= col < self.width):
            return False
        return state not in self.walls

    def states(self):
        return [
            (row, col)
            for row in range(self.height)
            for col in range(self.width)
            if self.is_valid_state((row, col))
        ]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        # R(s): recompensa de estar en el estado actual.
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]

        Recuerda:
        - las probabilidades dependen de si 'state' es resbaloso;
        - si golpea pared/borde, next_state = state.
        """
        # Un estado terminal es absorbente: no hay mas decisiones que tomar.
        if self.is_terminal(state):
            return [(state, 1.0)]

        # La dinamica depende del estado ACTUAL (piso normal vs resbaloso).
        if state in self.slippery_states:
            p_intended, p_dev = 0.60, 0.20
        else:
            p_intended, p_dev = 0.90, 0.05

        # Desviaciones perpendiculares a la acción elegida
        # (mismo truco que en 02_policy_iteration_gridworld.ipynb):
        # para action=(dr,dc), sus dos perpendiculares son (dc,dr) y (-dc,-dr).
        perp1 = (action[1], action[0])
        perp2 = (-action[1], -action[0])

        outcomes = [
            (action, p_intended),
            (perp1, p_dev),
            (perp2, p_dev),
        ]

        transitions = []
        for d, prob in outcomes:
            next_state = (state[0] + d[0], state[1] + d[1])
            if not self.is_valid_state(next_state):
                # Sale del grid o golpea una estanteria: se queda quieto.
                next_state = state
            transitions.append((next_state, prob))

        return transitions



### Validación minima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [ ]:
grid = WarehouseMDP()

S = grid.states()
print("Numero de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [3]:
def expected_next_value(grid, state, action, V):
    # sum_{s'} T(s,a,s') V(s')
    return sum(
        prob * V[next_state]
        for next_state, prob in grid.get_transition_probs(state, action)
    )


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    V = {state: 0.0 for state in grid.states()}

    for iteration in range(max_iter):
        V_new = V.copy()
        biggest_change = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                # Terminal absorbente: su valor es directamente su recompensa.
                V_new[state] = grid.get_reward(state)
            else:
                V_new[state] = grid.get_reward(state) + grid.gamma * max(
                    expected_next_value(grid, state, action, V)
                    for action in grid.actions
                )

            biggest_change = max(
                biggest_change, abs(V_new[state] - V[state])
            )

        V = V_new
        if biggest_change < threshold:
            break

    return V, iteration + 1


def extract_policy(grid, V):
    # pi*(s) = argmax_a sum T(s,a,s') V(s')
    policy = {}
    for state in grid.states():
        if grid.is_terminal(state):
            continue
        policy[state] = max(
            grid.actions,
            key=lambda action: expected_next_value(grid, state, action, V)
        )
    return policy



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [ ]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    V = {state: 0.0 for state in grid.states()}

    for iteration in range(max_iter):
        V_new = V.copy()
        biggest_change = 0.0

        for state in grid.states():
            if grid.is_terminal(state):
                V_new[state] = grid.get_reward(state)
            else:
                # La politica ya escogió la acción: NO hay max aqui.
                action = policy[state]
                V_new[state] = grid.get_reward(state) + grid.gamma * expected_next_value(
                    grid, state, action, V
                )

            biggest_change = max(
                biggest_change, abs(V_new[state] - V[state])
            )

        V = V_new
        if biggest_change < threshold:
            break

    return V, iteration + 1


def policy_improvement(grid, V):
    new_policy = {}
    for state in grid.states():
        if grid.is_terminal(state):
            continue
        new_policy[state] = max(
            grid.actions,
            key=lambda action: expected_next_value(grid, state, action, V)
        )
    return new_policy


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    # 1. Politica inicial arbitraria: RIGHT en todos los estados no terminales.
    initial_action = (0, 1)
    policy = {
        state: initial_action
        for state in grid.states()
        if not grid.is_terminal(state)
    }

    history = []

    for iteration in range(max_iter):
        # 2. Evaluación
        V, eval_iterations = policy_evaluation(grid, policy, threshold=threshold)

        # 3. Mejora
        new_policy = policy_improvement(grid, V)

        changed = sum(
            new_policy[state] != policy[state] for state in new_policy
        )

        history.append({
            "policy_iteration": iteration + 1,
            "evaluation_sweeps": eval_iterations,
            "changed_actions": changed,
        })

        policy = new_policy

        # 4. Repetir hasta estabilidad
        if changed == 0:
            break

    V, _ = policy_evaluation(grid, policy, threshold=threshold)

    return policy, V, history



## Parte 4 — Visualización y comparación


In [5]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [ ]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolitica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolitica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma politica óptima.")


=== VALUE ITERATION ===
Iteraciones: 20

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [{'policy_iteration': 1, 'evaluation_sweeps': 73, 'changed_actions': 19}, {'policy_iteration': 2, 'evaluation_sweeps': 61, 'changed_actions': 7}, {'policy_iteration': 3, 'evaluation_sweeps': 19, 'changed_actions': 1}, {'policy_iteration': 4, 'evaluation_sweeps': 22, 'changed_actions': 0}]

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +


## Parte 5 — Interpreta la politica

Antes de cambiar parametros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por que una recompensa menor podria ser óptima?
3. ¿En que estados el piso resbaloso cambia la decisión?
4. ¿Que papel cumple el costo por paso `-1`?
5. ¿Por que \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la politica **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Mas paciencia

Cambia:

```python
gamma = 0.99
```

¿La politica valora mas la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la politica desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


In [ ]:
# Verificación de la Pregunta 3

class WarehouseMDP_NoSlip(WarehouseMDP):
    """Mismo grid, mismas paredes/terminales/peligros, pero sin piso resbaloso"""

    def get_transition_probs(self, state, action):
        if self.is_terminal(state):
            return [(state, 1.0)]

        p_intended, p_dev = 0.90, 0.05

        perp1 = (action[1], action[0])
        perp2 = (-action[1], -action[0])
        outcomes = [(action, p_intended), (perp1, p_dev), (perp2, p_dev)]

        transitions = []
        for d, prob in outcomes:
            next_state = (state[0] + d[0], state[1] + d[1])
            if not self.is_valid_state(next_state):
                next_state = state
            transitions.append((next_state, prob))
        return transitions


g_slip = WarehouseMDP()
V_slip, _ = value_iteration(g_slip)
pi_slip = extract_policy(g_slip, V_slip)

g_noslip = WarehouseMDP_NoSlip()
V_noslip, _ = value_iteration(g_noslip)
pi_noslip = extract_policy(g_noslip, V_noslip)

print("Acción óptima en cada celda resbalosa (con piso resbaloso vs. sin piso resbaloso):")
for s in sorted(g_slip.slippery_states):
    same = pi_slip[s] == pi_noslip[s]
    print(f"  {s}: con resbalón -> {pi_slip[s]} | sin resbalón -> {pi_noslip[s]} | ¿cambia? {not same}")

diffs = sorted(s for s in pi_slip if pi_slip[s] != pi_noslip.get(s))
print("\nTotal de estados (en toda la grilla) donde la acción óptima cambia:", len(diffs))
print("Estados:", diffs)


Acción óptima en cada celda resbalosa (con piso resbaloso vs. sin piso resbaloso):
  (1, 2): con resbalón -> (1, 0) | sin resbalón -> (0, 1) | ¿cambia? True
  (2, 1): con resbalón -> (0, 1) | sin resbalón -> (0, 1) | ¿cambia? False
  (3, 3): con resbalón -> (-1, 0) | sin resbalón -> (-1, 0) | ¿cambia? False

Total de estados (en toda la grilla) donde la acción óptima cambia: 2
Estados: [(1, 2), (3, 1)]


### Respuestas — interpretación de la politica óptima

**1. Desde `START`, ¿el robot busca la entrega `+10` o prefiere la estación de carga `+2`?**

Con  los parametros base (`living_reward = -1.0`, `gamma = 0.9`, piso resbaloso con `p_intended = 0.60`), la politica óptima desde `START = (0,0)` es ir a la estación de carga `+2`, no a la entrega `+10`. Siguiendo la politica óptima, la trayectoria (asumiendo que la acción intentada siempre se cumple) es:

`(0,0) -> (0,1) -> (0,2) -> (1,2) -> (2,2)=CARGA +2`

es decir, solo 4 pasos.

**2. ¿Por que una recompensa menor podria ser óptima?**

Porque lo que se maximiza no es la recompensa terminal por si sola, sino la suma descontada de recompensas en el tiempo. Llegar a `+10` obliga a rodear la estanteria en `(0,3)` y pasar cerca (o por) la celda de peligro `-3` en `(1,4)`, lo que implica muchos mas pasos de costo `-1` (y descuento `gamma^t` reduciendo el valor de una recompensa lejana). El  costo acumulado de step + el riesgo de `-3` hacen que, en valor presente, `+2` cercano (`V(start) = -2.575`) sea mas atractivo que arriesgarse a buscar el `+10` lejano.

**3. ¿En que estados el piso resbaloso cambia la decisión?**

Comparando la politica óptima con una versión hipotetica del mismo mundo *sin* piso resbaloso (todas las celdas con `p_intended = 0.90`), la acción óptima cambia en la celda resbalosa `(1,2)`: en el mundo real (con resbalón) el robot prefiere ir DOWN directo a la carga `(2,2)`, mientras que sin resbalón preferiria seguir RIGHT hacia la entrega. La razón es que en `(1,2)` el riesgo de desviación es mayor (`p_dev = 0.20` en vez de `0.05`), asi que conviene tomar la ruta mas corta y selecccionar una acción cuyas desviaciones no sean catastróficas (al ir DOWN, incluso si se desvia, se desliza lateralmente por la fila 1, no hacia atras). Este cambio tambien se propaga a un estado vecino no resbaloso (`(3,1)`), mostrando que el  efecto del resbalón no queda aislado a la celda misma.

**4. ¿Que papel cumple el costo por paso `-1`?**

Actua como un "impuesto al tiempo": penaliza cualquier trayctoria larga, incentivando caminos cortos y desalentando explorar hacia recompensas lejanas aunque sean grandes. Es lo que hace que, en el caso base, `+2` cercano le gane a `+10` lejano. Si se reduce (Experimento A) o se aumenta la paciencia con `gamma` (Experimento C), ese balance cambia.

**5. ¿Por que `T(s,a,s')` ya no puede implementarse con las mismas probabilidades para todos los estados?**

Porque, a diferencia del GridWorld de `02_policy_iteration_gridworld.ipynb` (donde el ruido era igual en todo el grid), aqui la dinamica depende del estado actual: las celdas de piso resbaloso tienen `p_intended = 0.60` (y `0.20`/`0.20` de desviación) mientras que el piso normal tiene `p_intended = 0.90` (y `0.05`/`0.05`). Por eso `get_transition_probs` debe primero revisar si `state in self.slippery_states` antes de fijar las probabilidades — el "ruido" del movimiento ya no es una constante global sino una propiedad de cada celda.


In [ ]:
# Experimento A (Menos costo por paso (living_reward = -0.1))
import copy

def run_experiment(living_reward=None, gamma=None, p_intended_slip=None):
    """Crea una copia del grid con los parametros modificados y resuelve con Value Iteration."""
    g = WarehouseMDP()
    if living_reward is not None:
        g.living_reward = living_reward
    if gamma is not None:
        g.gamma = gamma
    if p_intended_slip is not None:
        p_dev = (1.0 - p_intended_slip) / 2.0

        def get_transition_probs(state, action, g=g, p_intended_slip=p_intended_slip, p_dev=p_dev):
            if g.is_terminal(state):
                return [(state, 1.0)]
            if state in g.slippery_states:
                p_intended, dev = p_intended_slip, p_dev
            else:
                p_intended, dev = 0.90, 0.05
            perp1 = (action[1], action[0])
            perp2 = (-action[1], -action[0])
            outcomes = [(action, p_intended), (perp1, dev), (perp2, dev)]
            transitions = []
            for d, prob in outcomes:
                next_state = (state[0] + d[0], state[1] + d[1])
                if not g.is_valid_state(next_state):
                    next_state = state
                transitions.append((next_state, prob))
            return transitions

        g.get_transition_probs = get_transition_probs

    V, n_iter = value_iteration(g)
    policy = extract_policy(g, V)
    return g, V, policy, n_iter


def rollout_terminal(grid, policy, max_steps=100):
    """Sigue la acción intencionada (determinista) desde START y regresa el terminal alcanzado."""
    s = grid.start
    for _ in range(max_steps):
        if grid.is_terminal(s):
            return s
        a = policy[s]
        ns = (s[0] + a[0], s[1] + a[1])
        if not grid.is_valid_state(ns):
            ns = s
        if ns == s:
            return None
        s = ns
    return None


g_A, V_A, pi_A, n_A = run_experiment(living_reward=-0.1)

print("EXPERIMENTO A: living_reward = -0.1")
print("Iteraciones:", n_A)
print("V(start) =", round(V_A[g_A.start], 3))
print("Acción óptima en START:", pi_A[g_A.start])
print("Terminal al que conduce la politica desde START:", rollout_terminal(g_A, pi_A))
print()
print_values(g_A, V_A)
print()
print_policy(g_A, pi_A)


EXPERIMENTO A: living_reward = -0.1
Iteraciones: 26
V(start) = 1.649
Acción óptima en START: (0, 1)
Terminal al que conduce la política desde START: (0, 5)

 +1.649 |  +1.992 |  +2.361 |   WALL   |  +8.591 | +10.000
 +1.358 |   WALL   |  +2.797 |  +3.911 |  +4.550 |  +8.591
 +1.259 |  +1.533 |  +2.000 |  +3.306 |   WALL   |  +7.537
 +1.243 |  +1.540 |  +2.040 |  +2.417 |  +2.026 | -10.000
 +0.865 |  -1.794 |   WALL   |  +2.026 |  +1.709 |  +0.874

 →  |  →  |  ↓  |  #  |  →  | +10
 ↑  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  →  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 


**Resultado del Experimento A:** con `living_reward = -0.1`, la politica óptima desde `START` cambia de objetivo: ahora la trayectoria greedy es

`(0,0) -> (0,1) -> (0,2) -> (1,2) -> (1,3) -> (1,4) -> (1,5) -> (0,5)=ENTREGA +10`

Es decir, el robot ahora si busca la entrega `+10`, incluso pasando por la celda de peligro `-3` en `(1,4)`. Esto confirma la predicción: al bajar el costo por paso, caminar mas pasos (o pasar por un peligro no terminal) ya no penaliza tanto, asi que el `+10` lejano vuelve a ser mas atractivo que el `+2` cercano. `V(start)` pasa de `= -2.575` (caso base) a `= +1.649`.


In [ ]:
# Experimento B (Piso muy resbaloso (0.60 -> 0.40, repartiendo el resto entre las desviaciones))
g_B, V_B, pi_B, n_B = run_experiment(p_intended_slip=0.40)

print("EXPERIMENTO B: p_intended (resbaloso) = 0.40")
print("Iteraciones:", n_B)
print("V(start) =", round(V_B[g_B.start], 3))
print("Acción óptima en START:", pi_B[g_B.start])
print("Terminal al que conduce la politica desde START:", rollout_terminal(g_B, pi_B))
print()
print_values(g_B, V_B)
print()
print_policy(g_B, pi_B)


EXPERIMENTO B: p_intended (resbaloso) = 0.40
Iteraciones: 22
V(start) = -2.706
Acción óptima en START: (0, 1)
Terminal al que conduce la política desde START: (2, 2)

 -2.706 |  -1.809 |  -0.797 |   WALL   |  +7.607 | +10.000
 -2.652 |   WALL   |  +0.395 |  +2.104 |  +3.670 |  +7.607
 -1.744 |  -0.670 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.831 |  -0.774 |  +0.534 |  -1.137 |  -2.152 | -10.000
 -2.785 |  -3.929 |   WALL   |  -2.152 |  -2.974 |  -4.041

 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 


**Resultado del Experimento B:** al empeorar el piso resbaloso (`p_intended = 0.40`, `p_dev = 0.30/0.30`), la politica desde `START` sigue prefiriendo la carga `+2` (mismo objetivo que el caso base) — tiene sentido, porque ya en el caso base el robot evitaba el riesgo del piso resbaloso; al aumentar aun mas el riesgo, la decisión conservadora se refuerza. Lo que si cambia es el valor: `V(start)` baja de `= -2.575` a `= -2.706`, porque incluso la ruta corta hacia `+2` pasa por una celda resbalosa `(1,2)` y ahora esa celda es mas impredecible.


In [ ]:
# Experimento C (Mas paciencia (gamma = 0.99))
g_C, V_C, pi_C, n_C = run_experiment(gamma=0.99)

print("EXPERIMENTO C: gamma = 0.99")
print("Iteraciones:", n_C)
print("V(start) =", round(V_C[g_C.start], 3))
print("Acción óptima en START:", pi_C[g_C.start])
print("Terminal al que conduce la politica desde START:", rollout_terminal(g_C, pi_C))
print()
print_values(g_C, V_C)
print()
print_policy(g_C, pi_C)


EXPERIMENTO C: gamma = 0.99
Iteraciones: 24
V(start) = -1.456
Acción óptima en START: (0, 1)
Terminal al que conduce la política desde START: (0, 5)

 -1.456 |  -0.310 |  +0.809 |   WALL   |  +8.601 | +10.000
 -2.179 |   WALL   |  +2.003 |  +4.118 |  +5.354 |  +8.601
 -1.081 |  +0.119 |  +2.000 |  +2.913 |   WALL   |  +7.395
 -1.606 |  -0.467 |  +0.799 |  +0.817 |  -0.359 | -10.000
 -2.752 |  -3.737 |   WALL   |  -0.359 |  -1.408 |  -2.892

 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 


**Resultado del Experimento C:** con `gamma = 0.99` (mucha mas paciencia), la politica desde `START` cambia hacia la entrega `+10`, igual que en el Experimento A pero por una razón distinta: aqui el costo por paso sigue siendo `-1`, pero al descontar casi nada el futuro (`gamma` cercano a 1), la recompensa grande y lejana `+10` pesa casi tanto como si fuera inmmediata, y termina superando a la recompnsa pequeña y cercana `+2`. `V(start)` sube de `= -2.575` (`gamma=0.9`) a `= -1.456` (`gamma=0.99`). Esto confirma que si, la politica valora mas la recompensa `+10` distante cuando el agente es mas paciente.


In [ ]:
# Bonus (valor aproximado de living_reward donde la politica desde START cambia) entre "ir a carga +2" e "intentar llegar a entrega +10"

def start_target(living_reward):
    g, V, policy, _ = run_experiment(living_reward=living_reward)
    return rollout_terminal(g, policy)

prev_target = None
coarse_switch = None
for lr in [round(-3.0 + 0.05 * i, 3) for i in range(60)]:
    t = start_target(lr)
    if prev_target is not None and t != prev_target:
        coarse_switch = (lr - 0.05, lr)
    prev_target = t

print("Cambio detectado (barrido grueso, paso 0.05) entre:", coarse_switch)

lo, hi = coarse_switch
fine_switch = None
lr = lo
prev_target = start_target(lo)
while lr <= hi + 1e-9:
    t = start_target(round(lr, 4))
    if t != prev_target:
        fine_switch = (round(lr - 0.002, 4), round(lr, 4))
        break
    prev_target = t
    lr += 0.002

print(f"Umbral aproximado de living_reward: entre {fine_switch[0]} y {fine_switch[1]}")
print(f"  living_reward = {fine_switch[0]} -> objetivo desde START:", start_target(fine_switch[0]))
print(f"  living_reward = {fine_switch[1]} -> objetivo desde START:", start_target(fine_switch[1]))


Cambio detectado (barrido grueso, paso 0.05) entre: (-0.8, -0.75)
Umbral aproximado de living_reward: entre -0.8 y -0.798
  living_reward = -0.8 -> objetivo desde START: (2, 2)
  living_reward = -0.798 -> objetivo desde START: (0, 5)


**Bonus:** barriendo `living_reward` (con `gamma = 0.9` y el piso resbaloso base) el objetivo desde `START` cambia de la carga `+2` a la entrega `+10` aproximadamente en `living_reward = -0.80`. Para `living_reward` mas negativo que ese umbral (pasos mas "caros"), conviene la carga `+2` cercana; para `living_reward` menos negativo (pasos mas "baratos"), conviene arriesgarse a buscar la entrega `+10`, mas lejana y bordeando la celda de peligro `-3`.
